# 04 · Feature-engineering experiment

**Plan (fixed 2026-11-11, before anything was run):** `docs/feature_engineering_plan.md`. This notebook tests whether cleaning the feature set makes the complete-linkage clustering less weighting-sensitive and more stable. It is an **experiment**: the approved primary model, `outputs/*`, the README and `reports/model_summary.md` are not changed. Associations only; screen time is a clustering feature, not a target. Medication use (`k4q23`) is parent-reported, not a diagnosis.

**Candidates** (equal weights W0, complete linkage, k = 2-6, the same five disjoint parts as notebook 03, seed 42):

| | Candidate | Features |
|---|---|---|
| A | no sex (`sc_sex` for profiling only) | 13 |
| B | `k4q22_r` + `k4q23` merged into one nominal `mh_care` (5 categories) | 13 |
| C | A + B + `k10q12/13/14` replaced by ordinal `amenity_count` (0-3) | 10 |
| D | C without `sleep_below_minimum` | 9 |

**Decision rules (judged at k = 3 only).** A candidate is "worth adopting" only if it meets all of 1-4; only the user decides whether to change the primary model.
1. More stable than the primary: mean pairwise ARI across the five full-data labellings >= 0.575 (primary: mean 0.525, min 0.302).
2. Less weighting-sensitive: ARI between its own W0 and W1 full-data labellings (part 0) is >= the minimum of its own replicate band (primary: 0.082).
3. No tiny clusters: no cluster under 1% of the 10,000-child sample in any part.
4. No single-feature split: no clustering feature has Cramér's V >= 0.8 with the cluster labels.
5. Silhouettes are reported but not compared across candidates (different distance definitions).
6. Other k values are informational; k is not chosen per candidate.

**Interpretations fixed before running (not changes to the plan):** rule 4 is judged on the part-0 full-data labelling (the analogue of the primary labelling); the other four parts are reported next to it. Cramér's V treats every feature as categorical (age as 18 integer levels) and is the plain, uncorrected statistic. Rule 3 uses the hierarchical sample labels (before assignment), as in notebook 03.

In [1]:
import gc, json, os, platform, time, tracemalloc
from contextlib import contextmanager
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import psutil
import scipy
import sklearn
from scipy.cluster.hierarchy import cophenet, cut_tree, fcluster, is_monotonic, is_valid_linkage, linkage
from scipy.spatial.distance import squareform
from scipy.stats import chi2_contingency, spearmanr
from sklearn.metrics import adjusted_rand_score, silhouette_samples

# ---- Configuration (identical to notebook 03 unless noted) -------------------
SEED = 42
SAMPLE_SIZE = int(os.environ.get("FE_SAMPLE_SIZE", 10_000))   # env override is for smoke tests only
N_PARTS = 5
STRATA_COLS = ["formtype", "screentime"]
K_RANGE = [2, 3, 4, 5, 6]                    # plan: k = 2-6
K_DECISION = 3
GOWER_BLOCK_ROWS = 500
ASSIGN_CHUNK_ROWS = 1_000
MIN_CLUSTER_SHARE = 0.01
sklearn.set_config(working_memory=256)

ROOT = Path.cwd()
DATA_PATH = ROOT / "datasets" / "preprocessed_data.csv"
OUT_DIR = Path(os.environ.get("FE_OUT", ROOT / "outputs" / "feature_engineering"))
CACHE_DIR = Path(os.environ["FE_CACHE"]) if os.environ.get("FE_CACHE") else None   # resume support (scratch only)
OUT_DIR.mkdir(parents=True, exist_ok=True)
if CACHE_DIR:
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
FULL_SIZE = SAMPLE_SIZE == 10_000

print("Python  ", platform.python_version())
for mod in (np, pd, scipy, sklearn, psutil):
    print(f"{mod.__name__:8s}", mod.__version__)
print(f"sample size {SAMPLE_SIZE:,} (full run: {FULL_SIZE}) | out dir {OUT_DIR}")

Python   3.12.7
numpy    1.26.4
pandas   2.2.2
scipy    1.13.1
sklearn  1.5.1
psutil   5.9.0
sample size 10,000 (full run: True) | out dir C:\Users\TeeRachen\OneDrive\เอกสาร\Data Mining Project\Children-Screentime-Clustering\outputs\feature_engineering


In [2]:
# ---- Memory and runtime monitoring (copied from notebook 03) ------------------
PROC = psutil.Process()
PERF_LOG = []


def rss_mib():
    return PROC.memory_info().rss / 2**20


def available_mib():
    return psutil.virtual_memory().available / 2**20


def require_memory(n_bytes, label, factor=1.5):
    need_mib = n_bytes * factor / 2**20
    if need_mib > available_mib():
        raise MemoryError(f"[{label}] needs ~{need_mib:,.0f} MiB (incl. x{factor} margin) but only "
                          f"{available_mib():,.0f} MiB is free. Close other applications and re-run.")


@contextmanager
def monitor(step, **info):
    gc.collect()
    rss_before = rss_mib()
    t0 = time.perf_counter()
    try:
        yield
    finally:
        PERF_LOG.append({"step": step, **info, "seconds": round(time.perf_counter() - t0, 3),
                         "rss_before_mib": round(rss_before, 1), "rss_after_mib": round(rss_mib(), 1),
                         "available_after_mib": round(available_mib(), 0)})

## 1 · Load and validate

In [3]:
df = pd.read_csv(DATA_PATH)
ID_COL = "hhid"
assert len(df) == 51_375 and df[ID_COL].is_unique
sleep_na = df["sleep_below_minimum"].isna()
assert sleep_na.equals(df["sc_age_lt4"].eq(1)), "structural missingness of sleep_below_minimum changed"

# Derived features used by the candidates
# mh_care: 1 none, 2 unmet need only, 3 treatment only, 4 medication without treatment received, 5 both
k22, k23 = df["k4q22_r"], df["k4q23"]
df["mh_care"] = np.select(
    [k22.eq(1) & k23.eq(1), k23.eq(1), k22.eq(1), k22.eq(2)], [5, 4, 3, 2], default=1).astype(int)
AMEN = ["k10q12", "k10q13", "k10q14"]
df["amenity_count"] = df[AMEN].eq(1).sum(axis=1)

mh_table = pd.crosstab(k22.map({1: "1 treatment received", 2: "2 needed, not received", 3: "3 not needed"}),
                       k23.map({1: "medication", 2: "no medication"}))
display(mh_table)
counts = df["mh_care"].value_counts().sort_index()
assert counts.to_dict() == {1: 41_253, 2: 978, 3: 3_789, 4: 1_924, 5: 3_431}, counts.to_dict()
assert df["amenity_count"].between(0, 3).all()
print("OK  mh_care counts match the plan (none 41,253; unmet need only 978; treatment only 3,789; "
      "medication without treatment 1,924; both 3,431)")
print("amenity_count distribution:", df["amenity_count"].value_counts().sort_index().to_dict())

k4q23,medication,no medication
k4q22_r,,
1 treatment received,3431,3789
"2 needed, not received",297,978
3 not needed,1627,41253


OK  mh_care counts match the plan (none 41,253; unmet need only 978; treatment only 3,789; medication without treatment 1,924; both 3,431)
amenity_count distribution: {0: 8430, 1: 8584, 2: 13182, 3: 21179}


## 2 · Feature specification and Gower helpers

Helpers are copied from `03_clustering.ipynb` (not imported). They were validated there against a pair-by-pair reference; Section 3 below re-validates them for the new feature types (5-category nominal, ordinal count) and re-creates the five parts.

In [4]:
FEATURE_TABLE = pd.DataFrame(
    [
        ("sc_age_years",        "general",     "numeric",     0, 17, np.nan, "Age in years"),
        ("sc_sex",              "general",     "binary_sym",  1,  2, np.nan, "Sex (1 male, 2 female)"),
        ("family_r",            "general",     "nominal",     1,  8, np.nan, "Family structure (8 types)"),
        ("blindness",           "health",      "binary_asym", 1,  2, 2,    "Vision difficulty even with glasses (1 yes, 2 no)"),
        ("k4q22_r",             "health",      "nominal",     1,  3, np.nan, "Mental health professional treatment, past 12 months"),
        ("k4q23",               "health",      "binary_asym", 1,  2, 2,    "Medication for emotions/concentration/behavior, past 12 months"),
        ("screentime",          "lifestyle",   "ordinal",     1,  5, np.nan, "Weekday screen time (<1 h ... 4+ h)"),
        ("sleep_below_minimum", "lifestyle",   "binary_sym",  0,  1, np.nan, "Sleep below age-specific recommended minimum"),
        ("k6q71_r",             "lifestyle",   "ordinal",     1,  4, np.nan, "Shows interest/curiosity in learning"),
        ("higrade",             "environment", "ordinal",     1,  3, np.nan, "Highest adult education"),
        ("foodsit",             "environment", "ordinal",     1,  4, np.nan, "Household food affordability"),
        ("k10q12",              "environment", "binary_sym",  1,  2, np.nan, "Park or playground in neighborhood"),
        ("k10q13",              "environment", "binary_sym",  1,  2, np.nan, "Recreation/community center in neighborhood"),
        ("k10q14",              "environment", "binary_sym",  1,  2, np.nan, "Library or bookmobile in neighborhood"),
    ],
    columns=["feature", "domain", "gower_type", "min", "max", "absent_code", "description"],
)
MH_ROW = ("mh_care", "health", "nominal", 1, 5, np.nan,
          "Mental-health care: 1 none, 2 unmet need only, 3 treatment only, 4 medication without treatment received, 5 both")
AMEN_ROW = ("amenity_count", "environment", "ordinal", 0, 3, np.nan,
            "Neighborhood amenities present (park, recreation center, library): count 0-3")
CANDIDATES = ["primary", "A_no_sex", "B_merged_mh", "C_combined", "D_C_no_sleep"]


def finalize(spec, weighting):
    spec = spec.reset_index(drop=True)
    spec["range"] = spec["max"] - spec["min"]
    if weighting == "W0":
        spec["weight"] = 1.0
    elif weighting == "W1":     # domain-balanced: every domain has total weight 1 (same rule as notebook 03)
        spec["weight"] = 1.0 / spec.groupby("domain")["feature"].transform("count")
    else:
        raise ValueError(weighting)
    return spec


def candidate_spec(name, weighting="W0"):
    spec = FEATURE_TABLE.copy()
    drop, add = [], []
    if name in ("A_no_sex", "C_combined", "D_C_no_sleep"):
        drop += ["sc_sex"]
    if name in ("B_merged_mh", "C_combined", "D_C_no_sleep"):
        drop += ["k4q22_r", "k4q23"]
        add.append(MH_ROW)
    if name in ("C_combined", "D_C_no_sleep"):
        drop += AMEN
        add.append(AMEN_ROW)
    if name == "D_C_no_sleep":
        drop += ["sleep_below_minimum"]
    if name not in CANDIDATES and not name.startswith("E_"):
        raise ValueError(name)
    spec = spec[~spec["feature"].isin(drop)]
    if add:
        spec = pd.concat([spec, pd.DataFrame(add, columns=FEATURE_TABLE.columns)], ignore_index=True)
    return finalize(spec, weighting)


def encode(data, spec):
    return data[spec["feature"].tolist()].to_numpy(dtype=np.float64)


def gower_cross(XA, XB, spec):
    num = np.zeros((XA.shape[0], XB.shape[0]))
    den = np.zeros_like(num)
    for k, f in enumerate(spec.itertuples(index=False)):
        a = XA[:, k][:, None]
        b = XB[:, k][None, :]
        if f.gower_type in ("numeric", "ordinal"):
            d = np.abs(a - b) / f.range
        else:
            d = (a != b).astype(np.float64)
        valid = ~(np.isnan(a) | np.isnan(b))
        if f.gower_type == "binary_asym":
            valid = valid & ~((a == f.absent_code) & (b == f.absent_code))
        num += f.weight * np.where(valid, d, 0.0)
        den += f.weight * valid
    if not (den > 0).all():
        raise ValueError("Zero Gower denominator: a pair has no comparable features")
    return num / den


def gower_condensed(X, spec, block_rows=GOWER_BLOCK_ROWS):
    n = X.shape[0]
    require_memory(n * (n - 1) // 2 * 8 + block_rows * n * 8 * 6, label=f"gower_condensed n={n}")
    out = np.empty(n * (n - 1) // 2, dtype=np.float64)
    pos = 0
    for start in range(0, n - 1, block_rows):
        stop = min(start + block_rows, n)
        block = gower_cross(X[start:stop], X[start:], spec)
        for r in range(stop - start):
            row = block[r, r + 1:]
            out[pos:pos + row.size] = row
            pos += row.size
    assert pos == out.size
    return out


def mean_distance_to_clusters(X_new, X_ref, label_sets, spec, chunk_rows=ASSIGN_CHUNK_ROWS):
    require_memory(chunk_rows * X_ref.shape[0] * 8 * 6, label="assignment chunk")
    prepared = {}
    for name, labels in label_sets.items():
        clusters = np.unique(labels)
        onehot = (labels[:, None] == clusters[None, :]).astype(np.float64)
        prepared[name] = (clusters, onehot, onehot.sum(axis=0))
    out = {name: np.empty((X_new.shape[0], len(c))) for name, (c, _, _) in prepared.items()}
    for s in range(0, X_new.shape[0], chunk_rows):
        block = gower_cross(X_new[s:s + chunk_rows], X_ref, spec)
        for name, (clusters, onehot, sizes) in prepared.items():
            out[name][s:s + chunk_rows] = (block @ onehot) / sizes
    return {name: (prepared[name][0], D) for name, D in out.items()}


def condensed_to_square_float32(d):
    n = int(round((1 + np.sqrt(1 + 8 * d.size)) / 2))
    require_memory(n * n * 4, label=f"square float32 n={n}")
    D = np.zeros((n, n), dtype=np.float32)
    pos = 0
    for i in range(n - 1):
        row = d[pos:pos + n - 1 - i]
        D[i, i + 1:] = row
        D[i + 1:, i] = row
        pos += row.size
    return D


def cophenetic_correlation(Z, d, chunk=5_000_000):
    c = cophenet(Z)
    mc, md = c.mean(), d.mean()
    sxy = sxx = syy = 0.0
    for s in range(0, d.size, chunk):
        a = c[s:s + chunk] - mc
        b = d[s:s + chunk] - md
        sxy += a @ b
        sxx += a @ a
        syy += b @ b
    return sxy / np.sqrt(sxx * syy)


def assign_from_distances(clusters, D):
    order = np.argsort(D, axis=1, kind="stable")
    rows = np.arange(D.shape[0])
    d1, d2 = D[rows, order[:, 0]], D[rows, order[:, 1]]
    return pd.DataFrame({"cluster": clusters[order[:, 0]], "second_best_cluster": clusters[order[:, 1]],
                         "assignment_margin": d2 - d1, "exact_tie": d2 == d1})


def stratified_disjoint_parts(data, strata_cols, part_size, n_parts, seed):
    rng = np.random.default_rng(seed)
    groups = data.groupby(strata_cols, sort=True).indices
    keys = sorted(groups)
    sizes = pd.Series([len(groups[k]) for k in keys], index=pd.MultiIndex.from_tuples(keys, names=strata_cols))
    exact = sizes * part_size / len(data)
    quota = np.floor(exact).astype(int)
    shortfall = part_size - quota.sum()
    remainders = (exact - quota).sort_values(ascending=False, kind="stable")
    quota.loc[remainders.index[:shortfall]] += 1
    assert quota.sum() == part_size and (quota * n_parts <= sizes).all()
    parts = [[] for _ in range(n_parts)]
    for k in keys:
        shuffled = rng.permutation(groups[k])
        q = int(quota.loc[k])
        for p in range(n_parts):
            parts[p].extend(shuffled[p * q:(p + 1) * q])
    parts = [rng.permutation(np.asarray(p)) for p in parts]
    return parts, quota

## 3 · Validation: helpers, new feature types, and the five parts

In [5]:
# (a) Pair-by-pair reference for all candidate specifications (both weightings), 120 real children incl. edge cases
def gower_pair_reference(x, y, spec):
    num = den = 0.0
    for k, f in enumerate(spec.itertuples(index=False)):
        a, b = x[k], y[k]
        if np.isnan(a) or np.isnan(b):
            continue
        if f.gower_type == "binary_asym" and a == f.absent_code and b == f.absent_code:
            continue
        d = abs(a - b) / f.range if f.gower_type in ("numeric", "ordinal") else float(a != b)
        num += f.weight * d
        den += f.weight
    return num / den


rng_val = np.random.default_rng(SEED)
special = np.concatenate([rng_val.choice(np.flatnonzero(sleep_na), 10, replace=False)]
                         + [rng_val.choice(np.flatnonzero(df["mh_care"].eq(c)), 10, replace=False) for c in (2, 3, 4, 5)])
val = df.iloc[np.concatenate([special, rng_val.choice(np.setdiff1d(np.arange(len(df)), special), 70, replace=False)])]
for name in CANDIDATES:
    for w in ("W0", "W1"):
        spec = candidate_spec(name, w)
        X = encode(val, spec)
        fast = gower_condensed(X, spec)
        ref = np.array([gower_pair_reference(X[i], X[j], spec) for i in range(len(X)) for j in range(i + 1, len(X))])
        assert np.max(np.abs(fast - ref)) < 1e-12, (name, w)
        assert np.isfinite(fast).all() and fast.min() >= 0 and fast.max() <= 1
    print(f"OK  {name:14s} {len(spec):2d} features | vectorised = pair-by-pair reference (W0 and W1)")

# (b) Hand-worked check for the new features: mh_care nominal (simple matching), amenity_count ordinal /3
spec_c = candidate_spec("C_combined")
base = dict(sc_age_years=10, family_r=1, blindness=2, mh_care=1, screentime=3, sleep_below_minimum=0,
            k6q71_r=1, higrade=3, foodsit=1, amenity_count=3)
t = pd.DataFrame([base, dict(base, mh_care=5, amenity_count=1, screentime=5)])
got = gower_cross(encode(t, spec_c), encode(t, spec_c), spec_c)[0, 1]
want = (0 + 0 + 0 + 1 + 2/4 + 0 + 0 + 0 + 0 + 2/3) / 9          # blindness: both "No" -> excluded (9 comparable features)
assert np.isclose(got, want, atol=1e-12), (got, want)
print(f"OK  hand-worked candidate C pair: computed {got:.6f} = {want:.6f}")

# (c) W1 weights per candidate
print(pd.DataFrame({n: candidate_spec(n, "W1").groupby("domain")["weight"].agg(["count", "first"]).apply(
    lambda r: f"{int(r['count'])} x {r['first']:.3f}", axis=1) for n in CANDIDATES}))

OK  primary        14 features | vectorised = pair-by-pair reference (W0 and W1)


OK  A_no_sex       13 features | vectorised = pair-by-pair reference (W0 and W1)


OK  B_merged_mh    13 features | vectorised = pair-by-pair reference (W0 and W1)


OK  C_combined     10 features | vectorised = pair-by-pair reference (W0 and W1)


OK  D_C_no_sleep    9 features | vectorised = pair-by-pair reference (W0 and W1)
OK  hand-worked candidate C pair: computed 0.240741 = 0.240741
               primary   A_no_sex B_merged_mh C_combined D_C_no_sleep
domain                                                               
environment  5 x 0.200  5 x 0.200   5 x 0.200  3 x 0.333    3 x 0.333
general      3 x 0.333  2 x 0.500   3 x 0.333  2 x 0.500    2 x 0.500
health       3 x 0.333  3 x 0.333   2 x 0.500  2 x 0.500    2 x 0.500
lifestyle    3 x 0.333  3 x 0.333   3 x 0.333  3 x 0.333    2 x 0.500


In [6]:
parts, quota = stratified_disjoint_parts(df, STRATA_COLS, SAMPLE_SIZE, N_PARTS, SEED)
assert all(len(p) == SAMPLE_SIZE for p in parts)
assert len(np.unique(np.concatenate(parts))) == N_PARTS * SAMPLE_SIZE
print(f"OK  {N_PARTS} disjoint parts of {SAMPLE_SIZE:,}")

# At the full size the parts must be exactly those of notebook 03 (stored in outputs/cluster_assignments.csv)
prev = pd.read_csv(ROOT / "outputs" / "cluster_assignments.csv")
assert (prev[ID_COL].to_numpy() == df[ID_COL].to_numpy()).all()
if FULL_SIZE:
    mine = np.full(len(df), -1)
    for p, idx in enumerate(parts):
        mine[idx] = p
    assert (mine == prev["replicate_part"].to_numpy()).all()
    print("OK  parts reproduce notebook 03's replicate_part exactly")
PRIMARY_REF = prev["cluster"].to_numpy()          # approved primary labelling (complete, W0, k = 3), read-only

OK  5 disjoint parts of 10,000
OK  parts reproduce notebook 03's replicate_part exactly


## 4 · Step 1: diagnostics (no clustering)

Everything below describes the data and the Gower distance only. Part 0 (10,000 children) is used for the distance contributions.

In [7]:
FEATS = FEATURE_TABLE["feature"].tolist()
TYPE = dict(zip(FEATURE_TABLE["feature"], FEATURE_TABLE["gower_type"]))


def cramers_v(a, b):
    ok = ~(pd.isna(a) | pd.isna(b))
    tab = pd.crosstab(np.asarray(a)[ok], np.asarray(b)[ok])
    if tab.shape[0] < 2 or tab.shape[1] < 2:
        return np.nan
    chi2 = chi2_contingency(tab, correction=False)[0]
    return float(np.sqrt(chi2 / (tab.values.sum() * (min(tab.shape) - 1))))


V = pd.DataFrame(np.nan, index=FEATS, columns=FEATS)
for a, b in combinations(FEATS, 2):
    V.loc[a, b] = V.loc[b, a] = cramers_v(df[a], df[b])
np.fill_diagonal(V.values, 1.0)

ORDERED = [f for f in FEATS if TYPE[f] != "nominal"]     # numeric, ordinal and binary codes can be rank-correlated
RHO = pd.DataFrame(np.nan, index=ORDERED, columns=ORDERED)
for a, b in combinations(ORDERED, 2):
    ok = df[a].notna() & df[b].notna()
    RHO.loc[a, b] = RHO.loc[b, a] = spearmanr(df.loc[ok, a], df.loc[ok, b])[0]
np.fill_diagonal(RHO.values, 1.0)

pd.set_option("display.width", 250, "display.max_columns", 30)
print("Cramér's V (all 14 features, categorical association; 0 = none, 1 = perfect):")
display(V.round(2))
print("Spearman rank correlation (ordered features only; binaries as 0/1-type codes):")
display(RHO.round(2))

pairs = pd.concat([
    V.where(np.triu(np.ones(V.shape, bool), 1)).stack().rename("cramers_v").reset_index().rename(columns={"level_0": "a", "level_1": "b"}),
], axis=1)
top_v = pairs.sort_values("cramers_v", ascending=False).head(12)
rho_pairs = RHO.where(np.triu(np.ones(RHO.shape, bool), 1)).stack().rename("spearman").reset_index().rename(columns={"level_0": "a", "level_1": "b"})
top_rho = rho_pairs.reindex(rho_pairs["spearman"].abs().sort_values(ascending=False).index).head(12)
print("Strongest pairs by Cramér's V:"); display(top_v.round(3))
print("Strongest pairs by |Spearman|:"); display(top_rho.round(3))
V.to_csv(OUT_DIR / "diag_cramers_v.csv")
RHO.to_csv(OUT_DIR / "diag_spearman.csv")

Cramér's V (all 14 features, categorical association; 0 = none, 1 = perfect):


,sc_age_years,sc_sex,family_r,blindness,k4q22_r,k4q23,screentime,sleep_below_minimum,k6q71_r,higrade,foodsit,k10q12,k10q13,k10q14
sc_age_years,1.00,0.02,0.08,0.05,0.17,0.22,0.30,0.19,0.19,0.03,0.03,0.04,0.02,0.02
sc_sex,0.02,1.00,0.01,0.01,0.04,0.01,0.03,0.00,0.04,0.00,0.01,0.01,0.00,0.02
family_r,0.08,0.01,1.00,0.04,0.12,0.11,0.09,0.14,0.08,0.19,0.15,0.05,0.03,0.04
blindness,0.05,0.01,0.04,1.00,0.04,0.03,0.05,0.02,0.05,0.06,0.08,0.02,0.01,0.02
k4q22_r,0.17,0.04,0.12,0.04,1.00,0.50,0.14,0.05,0.17,0.03,0.07,0.01,0.01,0.02
k4q23,0.22,0.01,0.11,0.03,0.50,1.00,0.17,0.03,0.21,0.02,0.06,0.03,0.01,0.01
screentime,0.30,0.03,0.09,0.05,0.14,0.17,1.00,0.13,0.17,0.05,0.09,0.05,0.03,0.04
sleep_below_minimum,0.19,0.00,0.14,0.02,0.05,0.03,0.13,1.00,0.06,0.11,0.14,0.03,0.03,0.03
k6q71_r,0.19,0.04,0.08,0.05,0.17,0.21,0.17,0.06,1.00,0.04,0.07,0.05,0.04,0.04
higrade,0.03,0.00,0.19,0.06,0.03,0.02,0.05,0.11,0.04,1.00,0.15,0.08,0.08,0.08


Spearman rank correlation (ordered features only; binaries as 0/1-type codes):


,sc_age_years,sc_sex,blindness,k4q23,screentime,sleep_below_minimum,k6q71_r,higrade,foodsit,k10q12,k10q13,k10q14
sc_age_years,1.00,0.00,-0.03,-0.22,0.50,0.03,0.32,-0.04,0.03,0.04,-0.02,0.00
sc_sex,0.00,1.00,-0.01,0.01,-0.03,-0.00,-0.04,0.00,-0.01,-0.01,0.00,-0.02
blindness,-0.03,-0.01,1.00,0.03,-0.04,-0.02,-0.03,0.06,-0.06,-0.02,-0.01,-0.02
k4q23,-0.22,0.01,0.03,1.00,-0.17,-0.03,-0.19,-0.02,-0.06,-0.03,-0.01,-0.01
screentime,0.50,-0.03,-0.04,-0.17,1.00,0.12,0.28,-0.07,0.15,0.05,0.02,0.04
sleep_below_minimum,0.03,-0.00,-0.02,-0.03,0.12,1.00,0.05,-0.11,0.14,0.03,0.03,0.03
k6q71_r,0.32,-0.04,-0.03,-0.19,0.28,0.05,1.00,-0.01,0.08,0.04,0.03,0.04
higrade,-0.04,0.00,0.06,-0.02,-0.07,-0.11,-0.01,1.00,-0.20,-0.08,-0.08,-0.08
foodsit,0.03,-0.01,-0.06,-0.06,0.15,0.14,0.08,-0.20,1.00,0.11,0.09,0.10
k10q12,0.04,-0.01,-0.02,-0.03,0.05,0.03,0.04,-0.08,0.11,1.00,0.40,0.48


Strongest pairs by Cramér's V:


,a,b,cramers_v
46,k4q22_r,k4q23,0.501
90,k10q13,k10q14,0.483
89,k10q12,k10q14,0.482
88,k10q12,k10q13,0.395
5,sc_age_years,screentime,0.302
4,sc_age_years,k4q23,0.222
57,k4q23,k6q71_r,0.213
7,sc_age_years,k6q71_r,0.193
6,sc_age_years,sleep_below_minimum,0.191
31,family_r,higrade,0.190


Strongest pairs by |Spearman|:


,a,b,spearman
3,sc_age_years,screentime,0.497
65,k10q13,k10q14,0.483
64,k10q12,k10q14,0.482
63,k10q12,k10q13,0.395
5,sc_age_years,k6q71_r,0.319
39,screentime,k6q71_r,0.278
2,sc_age_years,k4q23,-0.215
56,higrade,foodsit,-0.197
32,k4q23,k6q71_r,-0.193
30,k4q23,screentime,-0.171


In [8]:
# Each feature's share of the total pairwise Gower distance (part 0, W0, primary 14-feature spec)
def gower_contributions(X, spec, block_rows=GOWER_BLOCK_ROWS):
    # Mean over all pairs of w*delta*d_k / sum(w*delta): the amount feature k adds to the pair's Gower distance.
    n, p = X.shape[0], len(spec)
    total = np.zeros(p)
    comparable = np.zeros(p)
    for start in range(0, n - 1, block_rows):
        stop = min(start + block_rows, n)
        A, B = X[start:stop], X[start:]
        m = A.shape[0]
        upper = np.arange(B.shape[0])[None, :] > np.arange(m)[:, None]      # each pair once
        terms, valids = [], []
        den = np.zeros((m, B.shape[0]))
        for k, f in enumerate(spec.itertuples(index=False)):
            a, b = A[:, k][:, None], B[:, k][None, :]
            d = np.abs(a - b) / f.range if f.gower_type in ("numeric", "ordinal") else (a != b).astype(float)
            valid = ~(np.isnan(a) | np.isnan(b))
            if f.gower_type == "binary_asym":
                valid = valid & ~((a == f.absent_code) & (b == f.absent_code))
            den += f.weight * valid
            terms.append(f.weight * np.where(valid, d, 0.0))
            comparable[k] += (valid & upper).sum()
        for k in range(p):
            total[k] += ((terms[k] / den) * upper).sum()
    n_pairs = n * (n - 1) / 2
    return total / n_pairs, comparable / n_pairs


spec14 = candidate_spec("primary")
part0 = df.iloc[parts[0]].reset_index(drop=True)
X0 = encode(part0, spec14)
contrib, comparable = gower_contributions(X0, spec14)
mean_d = gower_condensed(X0, spec14).mean()
assert np.isclose(contrib.sum(), mean_d, atol=1e-10), (contrib.sum(), mean_d)
share = pd.DataFrame({"feature": spec14["feature"], "type": spec14["gower_type"], "domain": spec14["domain"],
                      "mean_contribution": contrib, "share_of_total_distance_%": contrib / contrib.sum() * 100,
                      "share_of_pairs_comparable_%": comparable * 100}).sort_values("share_of_total_distance_%", ascending=False)
print(f"OK  contributions sum to the mean pairwise Gower distance ({mean_d:.4f})")
display(share.round(3).reset_index(drop=True))
share.to_csv(OUT_DIR / "diag_distance_share.csv", index=False)
domain_share = share.groupby("domain")["share_of_total_distance_%"].sum().round(1)
print("By domain (%):", domain_share.to_dict())
print("Environment features, k10q12-14 together:", round(share.loc[share['feature'].isin(AMEN), 'share_of_total_distance_%'].sum(), 1), "%")

OK  contributions sum to the mean pairwise Gower distance (0.3682)


,feature,type,domain,mean_contribution,share_of_total_distance_%,share_of_pairs_comparable_%
0,family_r,nominal,general,0.041,11.249,100.000
1,sc_sex,binary_sym,general,0.041,11.130,100.000
2,k10q13,binary_sym,environment,0.041,11.099,100.000
3,k10q14,binary_sym,environment,0.036,9.664,100.000
4,sleep_below_minimum,binary_sym,lifestyle,0.035,9.438,98.386
5,screentime,ordinal,lifestyle,0.031,8.421,100.000
6,k10q12,binary_sym,environment,0.029,7.913,100.000
7,sc_age_years,numeric,general,0.029,7.823,100.000
8,k4q22_r,nominal,health,0.023,6.268,100.000
9,k6q71_r,ordinal,lifestyle,0.020,5.554,100.000


By domain (%): {'environment': 35.6, 'general': 30.2, 'health': 10.8, 'lifestyle': 23.4}
Environment features, k10q12-14 together: 28.7 %


In [9]:
# Prevalence of the binary features (all children and part 0), as the share with the 'event' code
BIN = {"sc_sex": ("female (2)", 2), "blindness": ("vision difficulty (1)", 1), "k4q23": ("medication (1)", 1),
       "sleep_below_minimum": ("below age minimum (1)", 1), "k10q12": ("park present (1)", 1),
       "k10q13": ("recreation center present (1)", 1), "k10q14": ("library present (1)", 1)}
prev_rows = []
for f, (label, code) in BIN.items():
    for scope, d in (("all children", df), ("part 0", part0)):
        s = d[f].dropna()
        prev_rows.append({"feature": f, "event": label, "scope": scope, "n": len(s), "event_%": (s == code).mean() * 100})
prevalence = pd.DataFrame(prev_rows).pivot(index=["feature", "event"], columns="scope", values="event_%").round(1)
display(prevalence)
prevalence.to_csv(OUT_DIR / "diag_binary_prevalence.csv")

# Mental-health items (counts already asserted in Section 1)
display(mh_table)
mh_summary = pd.DataFrame({"mh_care category": ["1 none", "2 unmet need only", "3 treatment only",
                                               "4 medication without treatment received", "5 both"],
                           "n": counts.to_numpy(), "%": (counts / len(df) * 100).round(1).to_numpy()})
display(mh_summary)
mh_summary.to_csv(OUT_DIR / "diag_mh_care_counts.csv", index=False)
mh_assoc = cramers_v(df["k4q22_r"], df["k4q23"])
print(f"Cramér's V between k4q22_r and k4q23: {mh_assoc:.3f}")

,scope,all children,part 0
feature,event,,
blindness,vision difficulty (1),1.4,1.5
k10q12,park present (1),76.6,76.9
k10q13,recreation center present (1),47.2,47.1
k10q14,library present (1),67.9,68.2
k4q23,medication (1),10.4,10.6
sc_sex,female (2),48.7,48.8
sleep_below_minimum,below age minimum (1),31.4,31.5


k4q23,medication,no medication
k4q22_r,,
1 treatment received,3431,3789
"2 needed, not received",297,978
3 not needed,1627,41253


,mh_care category,n,%
0,1 none,41253,80.3
1,2 unmet need only,978,1.9
2,3 treatment only,3789,7.4
3,4 medication without treatment received,1924,3.7
4,5 both,3431,6.7


Cramér's V between k4q22_r and k4q23: 0.501


### Step 1 conclusion: no extra candidate (E)

Decided on 2026-11-11 **before any clustering was run**, from the diagnostics above (part 0 and all children):

- The two mental-health items overlap (Cramér's V 0.50), and 3 of 4 `k4q22_r` x `k4q23` cells are non-trivial: this is candidate B.
- The three neighborhood items are correlated (V 0.40-0.48) and together carry 28.7% of the total pairwise distance, about as much as the whole health domain plus half of lifestyle: this is candidate C.
- `sc_sex` carries 11.1% of the distance with a roughly 50/50 split and is unrelated to every other feature (V <= 0.04): this is candidate A.
- The largest remaining pair is age x screen time (Spearman 0.50). That is a real developmental pattern, not redundancy, and screen time must stay a feature. `family_r` also carries 11.2% of the distance, but its category labels have not been verified against the codebook, so collapsing it would be a guess.
- Nothing in the diagnostics points to a further change that is both justified and checkable, so **no extra candidate is added**: the candidate set is exactly A-D from the plan.

The `primary` run below re-runs the approved model through this notebook's pipeline. It validates the copied helpers (its k = 3 labelling must equal `outputs/cluster_assignments.csv`) and gives the primary's labellings at k = 2-6 for the comparisons.

## 5 · Run machinery

One run = Gower distances for the 10,000 children of a part, complete linkage, silhouette for k = 2-6, then assignment of the other 41,375 children to the cluster with the smallest mean Gower distance to its sample members (same rule as notebook 03). Results are cached in a scratch folder when `FE_CACHE` is set, so an interrupted run can resume; the cache is not part of the deliverables. Progress lines are appended to `outputs/feature_engineering/progress.md`.

In [10]:
import hashlib, pickle

PROGRESS = OUT_DIR / "progress.md"


def log_progress(msg):
    with open(PROGRESS, "a", encoding="utf-8") as fh:
        fh.write(f"- {time.strftime('%H:%M')} {msg}\n")
    print(msg)


def evaluate_cuts(D_sq, Z, n, tags):
    rows, cl_rows, labs = [], [], {}
    for k in K_RANGE:
        labels = fcluster(Z, t=k, criterion="maxclust")
        sizes = np.bincount(labels)[1:]
        tie_fallback = len(sizes) != k
        if tie_fallback:
            # Tied merge heights (discrete Gower distances) make a height cut return fewer than k clusters.
            # Fallback: undo exactly the last k-1 merges in linkage order (identical to the height cut when no ties).
            labels = cut_tree(Z, n_clusters=k).ravel() + 1
            sizes = np.bincount(labels)[1:]
        assert len(sizes) == k, f"cut at k={k} produced {len(sizes)} clusters"
        s = silhouette_samples(D_sq, labels, metric="precomputed")
        labs[k] = labels
        rows.append({**tags, "k": k, "mean_silhouette": s.mean(), "smallest_cluster": int(sizes.min()),
                     "smallest_cluster_share_%": sizes.min() / n * 100,
                     "largest_cluster_share_%": sizes.max() / n * 100,
                     "clusters_below_1pct": int((sizes < MIN_CLUSTER_SHARE * n).sum()), "tie_fallback_cut": tie_fallback,
                     "sizes": " ".join(map(str, sorted(sizes, reverse=True)))})
        for c, size in enumerate(sizes, start=1):
            cl_rows.append({**tags, "k": k, "cluster": c, "size": int(size), "mean_silhouette": s[labels == c].mean()})
    return pd.DataFrame(rows), pd.DataFrame(cl_rows), labs


def run_part(name, weighting, p, extra_spec=None):
    spec = candidate_spec(name, weighting) if extra_spec is None else extra_spec
    key = hashlib.md5((name + weighting + str(p) + str(SAMPLE_SIZE) + spec.to_csv()).encode()).hexdigest()[:12]
    cache_file = CACHE_DIR / f"{name}_{weighting}_p{p}_{key}.pkl" if CACHE_DIR else None
    if cache_file is not None and cache_file.exists():
        with open(cache_file, "rb") as fh:
            return pickle.load(fh)
    tags = {"candidate": name, "weighting": weighting, "part": p}
    idx = parts[p]
    X = encode(df.iloc[idx], spec)
    with monitor(f"gower {name} {weighting}", part=p):
        d = gower_condensed(X, spec)
    assert np.isfinite(d).all() and d.min() >= 0 and d.max() <= 1
    require_memory(d.nbytes, label=f"linkage {name}")
    with monitor(f"linkage {name} {weighting}", part=p):
        Z = linkage(d, method="complete")
    assert is_valid_linkage(Z) and is_monotonic(Z)
    coph = cophenetic_correlation(Z, d)
    D_sq = condensed_to_square_float32(d)
    del d
    gc.collect()
    with monitor(f"silhouette {name} {weighting}", part=p):
        ev, percl, labs = evaluate_cuts(D_sq, Z, len(idx), tags)
    del D_sq
    gc.collect()
    in_sample = np.zeros(len(df), dtype=bool)
    in_sample[idx] = True
    out_idx = np.flatnonzero(~in_sample)
    with monitor(f"assignment {name} {weighting}", part=p):
        D_by_k = mean_distance_to_clusters(encode(df.iloc[out_idx], spec), X, labs, spec)
    full, amb = {}, {}
    for k, labels in labs.items():
        asg = assign_from_distances(*D_by_k[k])
        lab = np.empty(len(df), dtype=np.int64)
        lab[idx] = labels
        lab[out_idx] = asg["cluster"].to_numpy()
        full[k] = lab
        amb[k] = float((asg["assignment_margin"] < 0.01).mean())
    ev["cophenetic"] = coph
    ev["assigned_margin_below_0.01_share"] = ev["k"].map(amb)
    res = {"eval": ev, "percl": percl, "sample": labs, "full": full, "coph": coph}
    if cache_file is not None:
        with open(cache_file, "wb") as fh:
            pickle.dump(res, fh)
    return res


RESULTS = {}      # (candidate, weighting, part) -> result


def run_candidate(name, parts_to_run=range(N_PARTS), weighting="W0"):
    for p in parts_to_run:
        t0 = time.perf_counter()
        RESULTS[(name, weighting, p)] = run_part(name, weighting, p)
        print(f"  {name} {weighting} part {p}: {time.perf_counter() - t0:5.0f} s | cophenetic "
              f"{RESULTS[(name, weighting, p)]['coph']:.3f} | free RAM {available_mib():,.0f} MiB")

## 6 · Runs: the primary (reference) and candidates A-D, W0, five parts

In [11]:
log_progress("Step 2/3 runs started: primary re-run (reference), then A, B, C, D (5 parts each, W0), then W1 on part 0.")
run_candidate("primary")
log_progress("primary (14 features, W0) finished on all five parts.")

Step 2/3 runs started: primary re-run (reference), then A, B, C, D (5 parts each, W0), then W1 on part 0.
  primary W0 part 0:     0 s | cophenetic 0.482 | free RAM 3,125 MiB
  primary W0 part 1:     0 s | cophenetic 0.511 | free RAM 3,123 MiB


  primary W0 part 2:     0 s | cophenetic 0.530 | free RAM 3,117 MiB
  primary W0 part 3:     0 s | cophenetic 0.425 | free RAM 3,115 MiB
  primary W0 part 4:     0 s | cophenetic 0.486 | free RAM 3,114 MiB
primary (14 features, W0) finished on all five parts.


In [12]:
# Validate the copied pipeline against the approved primary results (full size only)
prim = RESULTS[("primary", "W0", 0)]["full"][3]
ari_ref = adjusted_rand_score(prim, PRIMARY_REF)
print(f"ARI of this notebook's primary k=3 labelling vs outputs/cluster_assignments.csv: {ari_ref:.6f}")
old = pd.read_csv(ROOT / "outputs" / "stability_sensitivity.csv")
old_stab = old[(old["kind"] == "replicate_stability") & (old["linkage"] == "complete") & (old["k"] == 3)]["ari"]
new_pairs = [adjusted_rand_score(RESULTS[("primary", "W0", a)]["full"][3], RESULTS[("primary", "W0", b)]["full"][3])
             for a, b in combinations(range(N_PARTS), 2)]
print(f"primary k=3 pairwise ARI: this notebook mean {np.mean(new_pairs):.4f} min {np.min(new_pairs):.4f} | "
      f"notebook 03 mean {old_stab.mean():.4f} min {old_stab.min():.4f}")
if FULL_SIZE:
    assert ari_ref == 1.0, "pipeline does not reproduce the approved primary labelling"
    assert np.isclose(np.mean(new_pairs), old_stab.mean(), atol=1e-9)
    print("OK  the pipeline reproduces the approved primary results exactly")
log_progress("Validation: this notebook's primary k=3 labelling equals the approved one (ARI 1.0) and the replicate ARIs match notebook 03."
             if FULL_SIZE else "Smoke test only: validation against approved results skipped (sample size != 10,000).")

ARI of this notebook's primary k=3 labelling vs outputs/cluster_assignments.csv: 1.000000
primary k=3 pairwise ARI: this notebook mean 0.5245 min 0.3019 | notebook 03 mean 0.5245 min 0.3019
OK  the pipeline reproduces the approved primary results exactly
Validation: this notebook's primary k=3 labelling equals the approved one (ARI 1.0) and the replicate ARIs match notebook 03.


In [13]:
run_candidate("A_no_sex")
log_progress("Candidate A (no sex, 13 features) finished on all five parts.")

  A_no_sex W0 part 0:     0 s | cophenetic 0.440 | free RAM 3,112 MiB
  A_no_sex W0 part 1:     0 s | cophenetic 0.426 | free RAM 3,110 MiB


  A_no_sex W0 part 2:     0 s | cophenetic 0.489 | free RAM 3,108 MiB
  A_no_sex W0 part 3:     0 s | cophenetic 0.569 | free RAM 3,106 MiB


  A_no_sex W0 part 4:     0 s | cophenetic 0.544 | free RAM 3,104 MiB
Candidate A (no sex, 13 features) finished on all five parts.


In [14]:
run_candidate("B_merged_mh")
log_progress("Candidate B (merged mental-health item, 13 features) finished on all five parts.")

  B_merged_mh W0 part 0:     0 s | cophenetic 0.483 | free RAM 3,101 MiB


  B_merged_mh W0 part 1:     0 s | cophenetic 0.447 | free RAM 3,099 MiB
  B_merged_mh W0 part 2:     0 s | cophenetic 0.431 | free RAM 3,097 MiB


  B_merged_mh W0 part 3:     0 s | cophenetic 0.489 | free RAM 3,092 MiB


  B_merged_mh W0 part 4:     0 s | cophenetic 0.527 | free RAM 3,088 MiB
Candidate B (merged mental-health item, 13 features) finished on all five parts.


In [15]:
run_candidate("C_combined")
log_progress("Candidate C (combined cleanup, 10 features) finished on all five parts.")

  C_combined W0 part 0:     0 s | cophenetic 0.464 | free RAM 3,084 MiB
  C_combined W0 part 1:     0 s | cophenetic 0.429 | free RAM 3,082 MiB


  C_combined W0 part 2:     0 s | cophenetic 0.466 | free RAM 3,080 MiB
  C_combined W0 part 3:     0 s | cophenetic 0.590 | free RAM 3,079 MiB


  C_combined W0 part 4:     0 s | cophenetic 0.509 | free RAM 3,073 MiB
Candidate C (combined cleanup, 10 features) finished on all five parts.


In [16]:
run_candidate("D_C_no_sleep")
log_progress("Candidate D (C without sleep, 9 features) finished on all five parts.")

  D_C_no_sleep W0 part 0:    67 s | cophenetic 0.490 | free RAM 3,095 MiB


  D_C_no_sleep W0 part 1:    67 s | cophenetic 0.656 | free RAM 3,016 MiB


  D_C_no_sleep W0 part 2:    66 s | cophenetic 0.563 | free RAM 2,966 MiB


  D_C_no_sleep W0 part 3:    66 s | cophenetic 0.653 | free RAM 3,081 MiB


  D_C_no_sleep W0 part 4:    66 s | cophenetic 0.677 | free RAM 3,070 MiB
Candidate D (C without sleep, 9 features) finished on all five parts.


### W1 (domain-balanced weights) on part 0, for the primary and each candidate

In [17]:
for name in CANDIDATES:
    t0 = time.perf_counter()
    RESULTS[(name, "W1", 0)] = run_part(name, "W1", 0)
    print(f"  {name} W1 part 0: {time.perf_counter() - t0:5.0f} s | free RAM {available_mib():,.0f} MiB")
log_progress("W1 runs (part 0) finished for the primary and candidates A-D.")

  primary W1 part 0:    91 s | free RAM 3,008 MiB


  A_no_sex W1 part 0:    86 s | free RAM 2,965 MiB


  B_merged_mh W1 part 0:    86 s | free RAM 3,005 MiB


  C_combined W1 part 0:    72 s | free RAM 3,065 MiB


  D_C_no_sleep W1 part 0:    67 s | free RAM 3,096 MiB
W1 runs (part 0) finished for the primary and candidates A-D.


## 7 · Evaluation

All numbers below are for **complete linkage, W0** unless stated. Silhouettes are each candidate's own (different distance definitions) and are **not compared across candidates** (rule 5); they are listed for completeness.

In [18]:
w0_eval = pd.concat([RESULTS[(n, "W0", p)]["eval"] for n in CANDIDATES for p in range(N_PARTS)], ignore_index=True)
w1_eval = pd.concat([RESULTS[(n, "W1", 0)]["eval"] for n in CANDIDATES], ignore_index=True)
all_eval = pd.concat([w0_eval, w1_eval], ignore_index=True)

print("Part 0 silhouette (own distance definition; NOT comparable across candidates) and sample sizes, W0:")
display(w0_eval[w0_eval["part"] == 0].pivot(index="candidate", columns="k", values="mean_silhouette").loc[CANDIDATES].round(3))
display(w0_eval[(w0_eval["part"] == 0) & (w0_eval["k"] == K_DECISION)].set_index("candidate")
        .loc[CANDIDATES, ["sizes", "smallest_cluster", "clusters_below_1pct", "cophenetic"]])
print("Smallest cluster in the 10,000-child sample, across the five parts:")
display(w0_eval.pivot_table(index="candidate", columns="k", values="smallest_cluster", aggfunc="min").loc[CANDIDATES])
print("Number of parts with a cluster under 1% of the sample:")
display(w0_eval.assign(flag=w0_eval["clusters_below_1pct"] > 0).pivot_table(index="candidate", columns="k", values="flag", aggfunc="sum").loc[CANDIDATES])

Part 0 silhouette (own distance definition; NOT comparable across candidates) and sample sizes, W0:


k,2,3,4,5,6
candidate,,,,,
primary,0.199,0.208,0.122,0.092,0.115
A_no_sex,0.067,0.111,0.098,0.114,0.119
B_merged_mh,0.234,0.185,0.125,0.123,0.118
C_combined,0.086,0.126,0.124,0.125,0.130
D_C_no_sleep,0.141,0.121,0.114,0.108,0.110


,sizes,smallest_cluster,clusters_below_1pct,cophenetic
candidate,,,,
primary,6833 2594 573,573,0,0.481673
A_no_sex,4274 3696 2030,2030,0,0.440130
B_merged_mh,5322 3119 1559,1559,0,0.483058
C_combined,4003 3609 2388,2388,0,0.463524
D_C_no_sleep,5898 3482 620,620,0,0.490324


Smallest cluster in the 10,000-child sample, across the five parts:


k,2,3,4,5,6
candidate,,,,,
primary,2297,271,271,151,151
A_no_sex,1728,193,109,109,109
B_merged_mh,2479,907,907,288,244
C_combined,2285,37,37,37,37
D_C_no_sleep,2107,51,24,23,23


Number of parts with a cluster under 1% of the sample:


k,2,3,4,5,6
candidate,,,,,
primary,0,0,0,0,0
A_no_sex,0,0,0,0,0
B_merged_mh,0,0,0,0,0
C_combined,0,1,2,2,2
D_C_no_sleep,0,2,4,4,4


In [19]:
# Pairwise ARI across the five full-data labellings (10 pairs) for every candidate and k
stab = pd.DataFrame([
    {"candidate": n, "k": k, "part_a": a, "part_b": b,
     "ari": adjusted_rand_score(RESULTS[(n, "W0", a)]["full"][k], RESULTS[(n, "W0", b)]["full"][k])}
    for n in CANDIDATES for k in K_RANGE for a, b in combinations(range(N_PARTS), 2)])
band = stab.groupby(["candidate", "k"])["ari"].agg(ari_min="min", ari_mean="mean", ari_max="max")
print("Replicate stability: pairwise ARI, 10 pairs (rule 1 uses the k = 3 mean):")
display(band.unstack("k").loc[CANDIDATES].round(3))

# ARI against the primary labelling (part 0 full-data labelling of each; informational except as context)
vs_primary = pd.DataFrame([
    {"candidate": n, "k": k, "ari_vs_primary": adjusted_rand_score(RESULTS[(n, "W0", 0)]["full"][k], RESULTS[("primary", "W0", 0)]["full"][k])}
    for n in CANDIDATES[1:] for k in K_RANGE])
print("ARI of each candidate's part-0 full-data labelling vs the primary's (same k):")
display(vs_primary.pivot(index="candidate", columns="k", values="ari_vs_primary").loc[CANDIDATES[1:]].round(3))

Replicate stability: pairwise ARI, 10 pairs (rule 1 uses the k = 3 mean):


ari_min                             ari_mean                             ari_max                            
k                  2      3      4      5      6        2      3      4      5      6       2      3      4      5      6
candidate                                                                                                                
primary        0.263  0.302  0.286  0.217  0.193    0.558  0.525  0.350  0.319  0.334   0.760  0.805  0.510  0.424  0.516
A_no_sex      -0.011  0.071  0.135  0.213  0.231    0.100  0.233  0.318  0.413  0.378   0.366  0.472  0.666  0.701  0.547
B_merged_mh   -0.000  0.209  0.283  0.331  0.266    0.322  0.354  0.428  0.452  0.422   0.784  0.539  0.600  0.610  0.602
C_combined     0.015  0.139  0.093  0.176  0.166    0.226  0.281  0.250  0.289  0.319   0.746  0.677  0.556  0.443  0.543
D_C_no_sleep   0.252  0.190  0.227  0.168  0.199    0.579  0.493  0.512  0.437  0.363   0.893  0.890  0.878  0.870  0.687

ARI of each candidate's part-0 full-data labelling vs the primary's (same k):


k,2,3,4,5,6
candidate,,,,,
A_no_sex,0.213,0.158,0.211,0.308,0.278
B_merged_mh,0.615,0.446,0.445,0.367,0.498
C_combined,-0.009,0.018,0.081,0.070,0.106
D_C_no_sleep,-0.013,0.043,0.039,0.047,0.035


In [20]:
# W0 vs W1 on the full data (part 0), against the candidate's own replicate band
w1_rows = []
for n in CANDIDATES:
    for k in K_RANGE:
        ari = adjusted_rand_score(RESULTS[(n, "W0", 0)]["full"][k], RESULTS[(n, "W1", 0)]["full"][k])
        lo, mean_, hi = band.loc[(n, k)]
        w1_rows.append({"candidate": n, "k": k, "ari_w0_vs_w1": ari, "band_min": lo, "band_mean": mean_,
                        "inside_band": bool(ari >= lo)})
w1_table = pd.DataFrame(w1_rows)
print("W0 vs W1 (full data, part 0) against each candidate's own replicate band:")
display(w1_table.pivot(index="candidate", columns="k", values="ari_w0_vs_w1").loc[CANDIDATES].round(3))
display(w1_table[w1_table["k"] == K_DECISION].set_index("candidate").loc[CANDIDATES].round(3))
if FULL_SIZE:
    old_w1 = old[(old["kind"] == "sensitivity") & (old["comparison"] == "W1 vs primary") & (old["k"] == 3)]["ari"].iloc[0]
    here = w1_table[(w1_table["candidate"] == "primary") & (w1_table["k"] == 3)]["ari_w0_vs_w1"].iloc[0]
    print(f"Check: primary W0 vs W1 here {here:.3f} vs ARI(W1, primary) in notebook 03: {old_w1:.3f}")
    assert np.isclose(here, old_w1, atol=1e-9)

W0 vs W1 (full data, part 0) against each candidate's own replicate band:


k,2,3,4,5,6
candidate,,,,,
primary,0.011,0.082,0.184,0.077,0.391
A_no_sex,0.007,0.021,0.059,0.086,0.177
B_merged_mh,0.007,0.009,0.019,0.015,0.013
C_combined,0.249,0.121,0.398,0.366,0.309
D_C_no_sleep,0.169,0.085,0.117,0.134,0.157


,k,ari_w0_vs_w1,band_min,band_mean,inside_band
candidate,,,,,
primary,3,0.082,0.302,0.525,False
A_no_sex,3,0.021,0.071,0.233,False
B_merged_mh,3,0.009,0.209,0.354,False
C_combined,3,0.121,0.139,0.281,False
D_C_no_sleep,3,0.085,0.190,0.493,False


Check: primary W0 vs W1 here 0.082 vs ARI(W1, primary) in notebook 03: 0.082


In [21]:
# Rule 4: Cramér's V between each clustering feature and the cluster labels (all 51,375 children)
def feature_v(labels, spec):
    return {f: cramers_v(df[f].to_numpy(), labels) for f in spec["feature"]}


v_rows = []
for n in CANDIDATES:
    spec = candidate_spec(n)
    for p in range(N_PARTS):
        for f, v in feature_v(RESULTS[(n, "W0", p)]["full"][K_DECISION], spec).items():
            v_rows.append({"candidate": n, "part": p, "k": K_DECISION, "feature": f, "cramers_v": v})
    for k in K_RANGE:                                           # informational: part 0, all k
        if k != K_DECISION:
            for f, v in feature_v(RESULTS[(n, "W0", 0)]["full"][k], spec).items():
                v_rows.append({"candidate": n, "part": 0, "k": k, "feature": f, "cramers_v": v})
vtab = pd.DataFrame(v_rows)
top_v = (vtab[(vtab["k"] == K_DECISION)].sort_values("cramers_v", ascending=False)
         .groupby(["candidate", "part"], sort=False).head(1).sort_values(["candidate", "part"]))
print("Largest Cramér's V between a clustering feature and the k = 3 cluster labels, per part:")
display(top_v[["candidate", "part", "feature", "cramers_v"]].round(3).set_index(["candidate", "part"]))
print("Part 0, k = 3: Cramér's V for every feature")
display(vtab[(vtab["part"] == 0) & (vtab["k"] == K_DECISION)].pivot(index="feature", columns="candidate", values="cramers_v")[CANDIDATES].round(2))

Largest Cramér's V between a clustering feature and the k = 3 cluster labels, per part:


feature  cramers_v
candidate    part                                
A_no_sex     0                  k10q13      0.864
             1     sleep_below_minimum      0.644
             2                  k10q12      0.867
             3                  k10q12      0.818
             4                  k10q13      0.710
B_merged_mh  0                  k10q13      0.862
             1                  sc_sex      0.839
             2                  k10q13      0.764
             3                  k10q14      0.820
             4                  k10q14      0.761
C_combined   0     sleep_below_minimum      0.934
             1            sc_age_years      0.401
             2     sleep_below_minimum      0.918
             3     sleep_below_minimum      0.663
             4                 mh_care      0.533
D_C_no_sleep 0                 mh_care      0.523
             1                 mh_care      0.605
             2                 mh_care      0.953
             3                 mh_care      0.937
             4                 mh_care      0.475
primary      0                  k10q14      0.837
             1                  k10q14      0.772
             2                  k10q14      0.749
             3                  k10q14      0.735
             4                  k10q14      0.731

Part 0, k = 3: Cramér's V for every feature


candidate,primary,A_no_sex,B_merged_mh,C_combined,D_C_no_sleep
feature,,,,,
amenity_count,NaN,NaN,NaN,0.06,0.06
blindness,0.03,0.01,0.02,0.03,0.16
family_r,0.11,0.06,0.04,0.17,0.26
foodsit,0.10,0.06,0.08,0.12,0.13
higrade,0.06,0.07,0.06,0.08,0.11
k10q12,0.65,0.45,0.50,NaN,NaN
k10q13,0.59,0.86,0.86,NaN,NaN
k10q14,0.84,0.52,0.60,NaN,NaN
k4q22_r,0.37,0.22,NaN,NaN,NaN


## 8 · Decision-rule table (k = 3)

Thresholds exactly as fixed in the plan. The `primary` row is the reference, not a candidate.

In [22]:
RULE1_MIN_MEAN_ARI, RULE4_MAX_V = 0.575, 0.8
rows = []
for n in CANDIDATES:
    b = band.loc[(n, K_DECISION)]
    w = w1_table[(w1_table["candidate"] == n) & (w1_table["k"] == K_DECISION)].iloc[0]
    sm = w0_eval[(w0_eval["candidate"] == n) & (w0_eval["k"] == K_DECISION)]
    tv = top_v[(top_v["candidate"] == n)]
    v0 = tv[tv["part"] == 0].iloc[0]
    r = {"candidate": n, "features": len(candidate_spec(n)),
         "mean_pairwise_ARI": b["ari_mean"], "min_pairwise_ARI": b["ari_min"], "max_pairwise_ARI": b["ari_max"],
         "R1_stable (mean >= 0.575)": bool(b["ari_mean"] >= RULE1_MIN_MEAN_ARI),
         "ARI_W0_vs_W1": w["ari_w0_vs_w1"], "own_band_min": w["band_min"],
         "R2_weighting_ok (>= band min)": bool(w["ari_w0_vs_w1"] >= w["band_min"]),
         "smallest_cluster_any_part": int(sm["smallest_cluster"].min()),
         "smallest_share_any_part_%": float(sm["smallest_cluster_share_%"].min()),
         "R3_no_tiny_cluster (>= 1% in all parts)": bool((sm["clusters_below_1pct"] == 0).all()),
         "max_V_part0": v0["cramers_v"], "max_V_feature": v0["feature"], "max_V_any_part": tv["cramers_v"].max(),
         "R4_no_single_feature_split (V < 0.8)": bool(v0["cramers_v"] < RULE4_MAX_V)}
    r["meets_rules_1-4"] = all(r[c] for c in r if c.startswith("R"))
    rows.append(r)
decision = pd.DataFrame(rows).set_index("candidate")
decision["verdict"] = np.where(decision.index == "primary", "reference",
                               np.where(decision["meets_rules_1-4"], "worth adopting (user decides)", "tested, not adopted"))
with pd.option_context("display.max_colwidth", None):
    display(decision.T)

candidate,primary,A_no_sex,B_merged_mh,C_combined,D_C_no_sleep
features,14,13,13,10,9
mean_pairwise_ARI,0.524515,0.233475,0.353718,0.281415,0.493131
min_pairwise_ARI,0.301883,0.070847,0.209463,0.139081,0.189694
max_pairwise_ARI,0.805311,0.472036,0.538842,0.676716,0.889714
R1_stable (mean >= 0.575),False,False,False,False,False
ARI_W0_vs_W1,0.08231,0.020962,0.008768,0.120588,0.08489
own_band_min,0.301883,0.070847,0.209463,0.139081,0.189694
R2_weighting_ok (>= band min),False,False,False,False,False
smallest_cluster_any_part,271,193,907,37,51
smallest_share_any_part_%,2.71,1.93,9.07,0.37,0.51


In [23]:
# Profiles at k = 3: part 0 full-data labelling, clusters renumbered by size (1 = largest), all 51,375 children.
# Associations only; medication use is parent-reported, not a diagnosis.
W = df["fwc"].to_numpy(dtype=float)
STATS = {"mean age (years)": df["sc_age_years"].astype(float),
         "% female": df["sc_sex"].eq(2) * 100.0,
         "% vision difficulty": df["blindness"].eq(1) * 100.0,
         "% mental-health treatment received": df["k4q22_r"].eq(1) * 100.0,
         "% mental-health care needed, not received": df["k4q22_r"].eq(2) * 100.0,
         "% medication for emotions/concentration/behavior": df["k4q23"].eq(1) * 100.0,
         "mean screen-time code (1-5)": df["screentime"].astype(float),
         "% screen time 4+ h": df["screentime"].eq(5) * 100.0,
         "% sleep below age minimum": df["sleep_below_minimum"] * 100.0,
         "% curiosity in learning 'always'": df["k6q71_r"].eq(1) * 100.0,
         "% highest adult education below high school": df["higrade"].eq(1) * 100.0,
         "% food sometimes/often not enough": df["foodsit"].ge(3) * 100.0,
         "mean neighborhood amenities present (0-3)": df["amenity_count"].astype(float),
         "% park or playground": df["k10q12"].eq(1) * 100.0,
         "% recreation/community center": df["k10q13"].eq(1) * 100.0,
         "% library": df["k10q14"].eq(1) * 100.0,
         "% questionnaire form T1 (ages 0-5)": df["formtype"].eq("T1") * 100.0}
MIN_N = 30
prof_rows = []
for n in CANDIDATES:
    lab = RESULTS[(n, "W0", 0)]["full"][K_DECISION]
    order = pd.Series(lab).value_counts().index.tolist()
    final = np.array([order.index(x) + 1 for x in lab])
    for c in [1, 2, 3, "all"]:
        mask = np.ones(len(df), bool) if c == "all" else final == c
        prof_rows.append({"candidate": n, "cluster": c, "statistic": "number of children", "n_used": int(mask.sum()),
                          "unweighted": float(mask.sum()), "weighted": np.nan})
        for stat, vals in STATS.items():
            v = vals.to_numpy(dtype=float)
            ok = mask & ~np.isnan(v)
            if ok.sum() < MIN_N:
                unw = wtd = np.nan
            else:
                unw, wtd = v[ok].mean(), np.average(v[ok], weights=W[ok])
            prof_rows.append({"candidate": n, "cluster": c, "statistic": stat, "n_used": int(ok.sum()),
                              "unweighted": unw, "weighted": wtd})
profiles = pd.DataFrame(prof_rows)
for n in CANDIDATES:
    wide = profiles[profiles["candidate"] == n].pivot(index="statistic", columns="cluster", values="unweighted")
    wide = wide.reindex(["number of children"] + list(STATS))[[1, 2, 3, "all"]]
    print(f"--- {n} (unweighted; 'weighted' by fwc is in the exported CSV) ---")
    display(wide.round(1))

--- primary (unweighted; 'weighted' by fwc is in the exported CSV) ---


cluster,1,2,3,all
statistic,,,,
number of children,33714.0,14477.0,3184.0,51375.0
mean age (years),9.6,8.9,13.2,9.7
% female,49.0,46.5,55.4,48.7
% vision difficulty,1.3,1.5,3.0,1.4
% mental-health treatment received,13.3,1.7,78.2,14.1
"% mental-health care needed, not received",2.4,1.1,9.5,2.5
% medication for emotions/concentration/behavior,9.0,2.3,62.2,10.4
mean screen-time code (1-5),3.2,3.1,4.1,3.2
% screen time 4+ h,22.5,21.4,51.3,24.0


--- A_no_sex (unweighted; 'weighted' by fwc is in the exported CSV) ---


cluster,1,2,3,all
statistic,,,,
number of children,23346.0,15639.0,12390.0,51375.0
mean age (years),8.9,12.1,7.9,9.7
% female,48.5,48.5,49.1,48.7
% vision difficulty,1.6,1.4,1.2,1.4
% mental-health treatment received,6.8,29.9,7.7,14.1
"% mental-health care needed, not received",1.9,3.7,2.0,2.5
% medication for emotions/concentration/behavior,6.3,20.8,5.1,10.4
mean screen-time code (1-5),3.1,3.8,2.8,3.2
% screen time 4+ h,20.7,33.9,17.7,24.0


--- B_merged_mh (unweighted; 'weighted' by fwc is in the exported CSV) ---


cluster,1,2,3,all
statistic,,,,
number of children,27512.0,14590.0,9273.0,51375.0
mean age (years),9.7,9.4,10.1,9.7
% female,55.1,7.2,94.9,48.7
% vision difficulty,1.3,1.3,2.0,1.4
% mental-health treatment received,14.1,11.6,17.7,14.1
"% mental-health care needed, not received",2.3,2.4,3.3,2.5
% medication for emotions/concentration/behavior,9.7,10.2,13.0,10.4
mean screen-time code (1-5),3.2,3.2,3.4,3.2
% screen time 4+ h,22.7,23.9,27.8,24.0


--- C_combined (unweighted; 'weighted' by fwc is in the exported CSV) ---


cluster,1,2,3,all
statistic,,,,
number of children,22933.0,14604.0,13838.0,51375.0
mean age (years),7.0,10.5,13.3,9.7
% female,49.4,48.6,47.5,48.7
% vision difficulty,1.0,1.7,1.9,1.4
% mental-health treatment received,2.2,15.5,32.1,14.1
"% mental-health care needed, not received",0.5,3.5,4.6,2.5
% medication for emotions/concentration/behavior,1.2,12.3,23.7,10.4
mean screen-time code (1-5),2.4,3.6,4.1,3.2
% screen time 4+ h,6.0,31.2,46.2,24.0


--- D_C_no_sleep (unweighted; 'weighted' by fwc is in the exported CSV) ---


cluster,1,2,3,all
statistic,,,,
number of children,29994.0,19095.0,2286.0,51375.0
mean age (years),7.0,13.4,12.9,9.7
% female,49.6,47.1,49.6,48.7
% vision difficulty,0.9,1.3,10.1,1.4
% mental-health treatment received,0.8,28.9,64.3,14.1
"% mental-health care needed, not received",0.4,3.2,24.1,2.5
% medication for emotions/concentration/behavior,0.5,19.3,66.7,10.4
mean screen-time code (1-5),2.7,4.0,4.1,3.2
% screen time 4+ h,10.3,41.9,53.5,24.0


## 9 · Exports

Written to `outputs/feature_engineering/` (no distance matrices; no child-level rows).

In [24]:
all_eval.to_csv(OUT_DIR / "candidate_evaluation.csv", index=False)
pd.concat([RESULTS[k]["percl"] for k in RESULTS], ignore_index=True).to_csv(OUT_DIR / "candidate_per_cluster_silhouette.csv", index=False)
stab.to_csv(OUT_DIR / "candidate_replicate_stability.csv", index=False)
band.reset_index().to_csv(OUT_DIR / "candidate_stability_band.csv", index=False)
vs_primary.to_csv(OUT_DIR / "candidate_vs_primary.csv", index=False)
w1_table.to_csv(OUT_DIR / "candidate_w0_vs_w1.csv", index=False)
vtab.to_csv(OUT_DIR / "candidate_cramers_v.csv", index=False)
decision.to_csv(OUT_DIR / "decision_rule_table.csv")
profiles.to_csv(OUT_DIR / "candidate_profiles_k3.csv", index=False)
pd.DataFrame(PERF_LOG).to_csv(OUT_DIR / "run_log.csv", index=False)
with open(OUT_DIR / "feature_engineering_config.json", "w", encoding="utf-8") as fh:
    json.dump({"seed": SEED, "sample_size": SAMPLE_SIZE, "n_parts": N_PARTS, "k_range": K_RANGE, "k_decision": K_DECISION,
               "linkage": "complete", "rules": {"R1_min_mean_ari": RULE1_MIN_MEAN_ARI, "R4_max_cramers_v": RULE4_MAX_V,
                                                 "R3_min_cluster_share": MIN_CLUSTER_SHARE},
               "candidates": {n: candidate_spec(n)[["feature", "domain", "gower_type", "min", "max"]].to_dict("records") for n in CANDIDATES},
               "packages": {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
                            "scipy": scipy.__version__, "scikit-learn": sklearn.__version__}}, fh, indent=2, default=str)
for f in sorted(OUT_DIR.iterdir()):
    print(f"{f.name:40s} {f.stat().st_size / 1024:8,.1f} KiB")
log_progress("Evaluation, decision-rule table, profiles and CSV exports written. Verdicts: "
             + "; ".join(f"{n}: {v}" for n, v in decision["verdict"].items()))
print(f"Total monitored time {pd.DataFrame(PERF_LOG)['seconds'].sum() / 60:.1f} min | process peak working set "
      f"{PROC.memory_info().peak_wset / 2**20:,.0f} MiB")

candidate_cramers_v.csv                      23.2 KiB
candidate_evaluation.csv                     16.7 KiB
candidate_per_cluster_silhouette.csv         21.4 KiB
candidate_profiles_k3.csv                    29.4 KiB
candidate_replicate_stability.csv             9.1 KiB
candidate_stability_band.csv                  1.8 KiB
candidate_vs_primary.csv                      0.7 KiB
candidate_w0_vs_w1.csv                        2.0 KiB
decision_rule_table.csv                       1.4 KiB
diag_binary_prevalence.csv                    0.3 KiB
diag_cramers_v.csv                            3.9 KiB
diag_distance_share.csv                       1.1 KiB
diag_mh_care_counts.csv                       0.2 KiB
diag_spearman.csv                             3.0 KiB
feature_engineering_config.json               9.6 KiB
progress.md                                   2.5 KiB
run_log.csv                                   2.1 KiB
Evaluation, decision-rule table, profiles and CSV exports written. Verdicts: prima